In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.stattools import durbin_watson

df = pd.read_csv(r"Book1.csv",parse_dates=["date"], dayfirst=True)
variables = ["CPIH excluding Energy", "Output growth", "Energy", "Sonia"]
df = df[variables].dropna()
df.head()

,CPIH excluding Energy,Output growth,Energy,Sonia
0,70.982,61.6,45.027,5.900455
1,71.157,61.9,44.947,5.985000
2,71.332,61.8,44.748,5.965068
3,71.682,62.6,44.542,5.961832
4,71.992,62.1,44.515,6.215380


In [10]:
# ── 2. ADF TEST (LEVELS) ──────────────────────────────────────────────────────
def adf_test(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    stat = "✓ stationary" if result[1] < 0.05 else "✗ non-stationary"
    print(f"  {name:30s}: ADF={result[0]:7.3f}, p={result[1]:.3f}  {stat}")
 
print("\n=== ADF Tests (levels) ===")
for col in variables:
    adf_test(df[col], col)


=== ADF Tests (levels) ===
  CPIH excluding Energy         : ADF=  1.687, p=0.998  ✗ non-stationary
  Output growth                 : ADF= -1.147, p=0.696  ✗ non-stationary
  Energy                        : ADF= -0.867, p=0.799  ✗ non-stationary
  Sonia                         : ADF= -2.221, p=0.199  ✗ non-stationary


In [11]:
# ── 3. FIRST DIFFERENCE + ADF TEST ───────────────────────────────────────────
df_diff = df.diff().dropna()
 
print("\n=== ADF Tests (first differences) ===")
for col in variables:
    adf_test(df_diff[col], col)
 
# ── 3b. SECOND DIFFERENCE FOR CPIH (still non-stationary at I(1)) ─────────────
df_diff2 = df_diff.copy()
df_diff2["CPIH excluding Energy"] = df["CPIH excluding Energy"].diff().diff().dropna()
df_diff2 = df_diff2.dropna()
 
print("\n=== ADF Test — CPIH (second difference) ===")
adf_test(df_diff2["CPIH excluding Energy"], "CPIH excluding Energy (d2)")


=== ADF Tests (first differences) ===
  CPIH excluding Energy         : ADF= -2.182, p=0.213  ✗ non-stationary
  Output growth                 : ADF=-10.814, p=0.000  ✓ stationary
  Energy                        : ADF= -5.646, p=0.000  ✓ stationary
  Sonia                         : ADF= -4.710, p=0.000  ✓ stationary

=== ADF Test — CPIH (second difference) ===
  CPIH excluding Energy (d2)    : ADF= -5.444, p=0.000  ✓ stationary


In [ ]:
# ── 4. LAG ORDER SELECTION ────────────────────────────────────────────────────
model = VAR(df_diff2)
lag_results = model.select_order(maxlags=12)
print("\n=== Lag Order Selection ===")
print(lag_results.summary())
optimal_lag = lag_results.aic
print(f"Optimal lag (AIC): {optimal_lag}")


=== Lag Order Selection ===
 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -2.086      -2.040      0.1242      -2.068
1       -2.566      -2.338     0.07684      -2.475
2       -2.760     -2.350*     0.06330      -2.597
3       -2.912      -2.320     0.05437      -2.676
4       -2.920      -2.145     0.05397      -2.611
5       -3.151      -2.194     0.04285      -2.769
6       -3.241      -2.103     0.03917      -2.787
7       -3.221      -1.900     0.03998      -2.695
8       -3.164      -1.661     0.04235      -2.565
9       -3.169      -1.484     0.04218      -2.498
10      -3.112      -1.245     0.04472      -2.368
11     -3.799*      -1.750    0.02254*     -2.982*
12      -3.782      -1.550     0.02298      -2.892
--------------------------------------------------
Optimal lag (AIC): 11


,CPIH excluding Energy,Output growth,Energy,Sonia
2,0.000,-0.1,-0.199,-0.019932
3,0.175,0.8,-0.206,-0.003237
4,-0.040,-0.5,-0.027,0.253548
5,-0.198,0.4,0.061,0.007282
6,-0.388,0.3,0.798,0.296512
...,...,...,...,...
344,-0.325,0.1,-0.201,-0.050002
345,0.369,-0.1,1.970,0.000997
346,-0.568,0.2,0.449,0.001080
347,0.637,0.1,0.786,-0.090379


In [13]:
chosen_lag = 3

fitted = model.fit(chosen_lag)
print("\n=== VAR Model Summary ===")
print(fitted.summary())


=== VAR Model Summary ===
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Thu, 09, Apr, 2026
Time:                     22:31:28
--------------------------------------------------------------------
No. of Equations:         4.00000    BIC:                   -2.40690
Nobs:                     344.000    HQIC:                  -2.75623
Log likelihood:          -1386.62    FPE:                  0.0504226
AIC:                     -2.98746    Det(Omega_mle):       0.0434696
--------------------------------------------------------------------
Results for equation CPIH excluding Energy
                              coefficient       std. error           t-stat            prob
-------------------------------------------------------------------------------------------
const                           -0.001238         0.017491           -0.071           0.944
L1.CPIH excluding Energy        -0.778131         0.052218       

In [ ]:
print("\n=== Durbin-Watson ===")
dw = durbin_watson(fitted.resid)
for col, stat in zip(variables, dw):
    flag = "✓" if 1.5 < stat < 2.5 else "⚠ check"
    print(f"  {col:30s}: {stat:.3f}  {flag}")
    



=== Durbin-Watson ===
  CPIH excluding Energy         : 2.103  ✓
  Output growth                 : 2.029  ✓
  Energy                        : 2.010  ✓
  Sonia                         : 1.920  ✓


In [16]:
print("\n=== Granger Causality — does Sonia cause CPIH & Output growth? ===")
gc = fitted.test_causality("CPIH excluding Energy", ["Sonia", "Energy"], kind="f")
print(gc.summary())


=== Granger Causality — does Sonia cause CPIH & Output growth? ===
Granger causality F-test. H_0: ['Sonia', 'Energy'] do not Granger-cause CPIH excluding Energy. Conclusion: fail to reject H_0 at 5% significance level.
Test statistic Critical value p-value     df   
-----------------------------------------------
         1.208          2.105   0.299 (6, 1324)
-----------------------------------------------
